# Active Discovery of Latent Dynamic Mediators

We are studying active causal discovery (of latent mediators) in systems where a controlled treatment $U$ influences an observed outcome $Y$ through a hidden mediating process $X$. The causal graph is $U → X → Y$ and $U →  Y$, with $X$ evolving dynamically over time according to a state space model (SSM). The central challenge is that $X$ is never directly observed. We are looking to answer the following questions:
1. Can we infer which latent process X is, from observations of Y and controlled inputs U?
2. Can we choose inputs U actively to resolve this identity faster?

[ADD DIAGRAM]

## Motivation
Every clinical treatment acts through a biological mechanism (which is almost never directly observable). If we can causally determine the biological mechanism, we can select the appropriate treatment without having to cycle through several rounds of ineffective treatments (for example in depression: [1])
Patients are heterogeneous (ergodicity as explained in [2])

[1] *Rush, A. John, et al. "Acute and longer-term outcomes in depressed outpatients requiring one or several treatment steps: a STAR* D report." American Journal of Psychiatry 163.11 (2006): 1905-1917.

[2] *Gu, Fei, Kristopher J. Preacher, and Emilio Ferrer. "A state space modeling approach to mediation analysis." Journal of Educational and Behavioral Statistics 39.2 (2014): 117-143.*

## Problem set up

**Assumptions**
- the system follows the Linear-Gaussian model
- the matrices in the system are infered/known and are stationary (no time dependency)

### Step 1: Infer the matrices
Since we are using synthetic data, we infer the matrices for the different potential mediating processes

[ADD DIAGRAM]

#### Example

### Step 2: Particle filter for determining posterior distribution
Use the matrices learnt in step 1 to run a Kalman Filter (KF) and (PF) to obtain the posterior of $X$

#### Kalman Filter

In [1]:
"""
Kalman Filter implementation for a simple linear system.

The system is defined as:
x_k = A*x_{k-1} + B*u_{k-1} + w_k
y_k = H*x_k + v_k
"""

import numpy as np

class KalmanFilter:
    def __init__(self, A, B, Q, H, R, x_0, P_0):
        """
        Initialize the Kalman Filter parameters.
        """
        self.A = A # State transition matrix
        self.B = B # Control input matrix
        self.Q = Q # Process noise covariance (for x)
        self.H = H # Measurement matrix
        self.R = R # Measurement noise covariance
        self.x_0 = x_0 # Initial state estimate
        self.P_0 = P_0 # Initial error covariance

    def predict(self, x_prev, P_prev, u_prev=None):
        """
        Predict the next state and error covariance.

        If u_prev is None, assumes zero control input.
        """
        control_effect = 0 if u_prev is None else self.B @ np.atleast_1d(u_prev)
        x_minus = self.A @ x_prev + control_effect
        P_minus = self.Q + self.A @ P_prev @ self.A.T
        return x_minus, P_minus

    def update(self, x_minus, P_minus, y):
        """
        Update the state estimate and error covariance with the new measurement.
        """
        # Kalman Gain
        K = P_minus@self.H.T@np.linalg.inv(self.H@P_minus@self.H.T+self.R)

        # Update state estimate and error covariance
        x = x_minus + K@(y - self.H@x_minus)
        P = (np.eye(len(P_minus))- K@self.H)@P_minus
        return x, P

#### Particle Filter

In [3]:
"""
Particle Filter implementation for state estimation.

The system is defined as:
x_k = A*x_{k-1} + B*u_{k-1} + w_k
y_k = H*x_k + v_k
"""
import numpy as np


class ParticleFilter:
    def __init__(self, num_particles, A, B, Q, H, R, x_0):
        """
        Initialize the Particle Filter parameters.
        """
        self.num_particles = num_particles
        self.A = A
        self.B = B
        self.Q = Q
        self.H = H
        self.R = R
        self.x_0 = x_0

        # Initialize particles and weights
        self.particles = np.random.multivariate_normal(x_0, Q, num_particles)
        self.weights = np.ones(num_particles) / num_particles

    def predict(self, u_prev):
        """
        Predict the next state of the particles.
        
        Generate process noise for each particle and update their states based on the system dynamics.
        """
        u_prev = np.atleast_1d(u_prev)

        # Generate process noise
        w = np.random.multivariate_normal(np.zeros(self.Q.shape[0]), self.Q, self.num_particles)
        
        # Update particle states
        for i in range(self.num_particles):
            self.particles[i] = self.A @ self.particles[i] + self.B @ u_prev + w[i]

    def update(self, y):
        """
        Update the weights of the particles based on the new measurement.

        Using observed output y_k and the weights w_{t-1}^i calculate the wight update, using
        w_t^i = p(y_k | x_k^i) * w_{t-1}^i
        where p(y_k | x_k^i) is the likelihood of the measurement given the particle's state, which can be calculated using the measurement model and the measurement noise covariance R.
        After updating the weights, normalize them so that they sum to 1.
        """
        for i in range(self.num_particles):
            # Calculate the likelihood of the measurement given the particle's state
            likelihood = self.gaussian_likelihood(y, self.H @ self.particles[i], self.R)
            self.weights[i] *= likelihood
        
        # Normalize weights
        self.weights /= np.sum(self.weights)

        # Resample particles based on their weights if N_eff is below a threshold of N/3
        N_eff = 1.0 / np.sum(self.weights**2)
        if N_eff < self.num_particles / 3:
            self.resample()

    def gaussian_likelihood(self, y, mean, cov):
        """
        Calculate the Gaussian likelihood of a measurement given a mean and covariance.
        """
        d = len(y)
        cov_inv = np.linalg.inv(cov)
        diff = y - mean
        exponent = -0.5 * diff.T @ cov_inv @ diff
        return (1.0 / np.sqrt((2 * np.pi)**d * np.linalg.det(cov))) * np.exp(exponent)

    def resample(self):
        """
        Resample the particles based on their weights.
        """
        # Perform systematic resampling
        indices = np.random.choice(self.num_particles, size=self.num_particles, p=self.weights)
        self.particles = self.particles[indices]
        self.weights = np.ones(self.num_particles) / self.num_particles

#### Example revisited

### Step 3: Matching posterior to best mediator candidate (model selection)
Match the posterior of $X$ to best mediator candidate

### Step 4: Active learning to select $U$

Use active learning to sample from $U$, to obtain the most information about $X$